In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: Qiskit Function od Qedmy
> **Note:** Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům plánů IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (prostřednictvím IBM Quantum Platform API). Jsou ve stavu preview verze a mohou se měnit.

## Přehled
Přestože se kvantové procesory (QPU) v posledních letech výrazně zlepšily, chyby způsobené hlukem a nedokonalostmi stávajícího hardwaru zůstávají klíčovou výzvou pro vývojáře kvantových algoritmů. Jak se obor přibližuje kvantovým výpočtům v měřítku užitečnosti, které nelze klasicky ověřit, stávají se řešení pro eliminaci šumu se zaručenou přesností čím dál důležitějšími. K překonání této výzvy vyvinula Qedma metodu Quantum Error Suppression and Error Mitigation (QESEM), která je bezešvě integrována na IBM Quantum Platform jako [Qiskit Function](/guides/functions).

S QESEM můžeš spouštět své kvantové obvody na hlučných QPU a získávat vysoce přesné výsledky bez chyb s vysoce efektivní spotřebou QPU času, blízkou fundamentálním limitům. QESEM k tomu využívá sadu proprietárních metod vyvinutých Qedmou pro charakterizaci a redukci chyb. Techniky redukce chyb zahrnují optimalizaci Gate, transpilaci s ohledem na šum, potlačení chyb (ES) a nestrannou mitigaci chyb (EM). Díky kombinaci těchto metod založených na charakterizaci mohou uživatelé dosáhnout spolehlivých výsledků bez chyb pro obecné kvantové obvody s velkým objemem, čímž se odemykají aplikace, které by jinak nebylo možné realizovat.

Úplný popis základních komponent i demonstraci v měřítku užitečnosti najdeš v článku [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Popis
Funkci QESEM od Qedmy můžeš použít ke snadnému odhadování a spouštění svých obvodů s potlačením a mitigací chyb, čímž dosáhneš větších objemů obvodů a vyšší přesnosti. Pro použití QESEM zadáš kvantový Circuit, sadu pozorovatelných veličin k měření, cílovou statistickou přesnost pro každou pozorovatelnou veličinu a zvolený QPU. Před spuštěním obvodu na cílovou přesnost můžeš odhadnout požadovaný čas QPU na základě analytického výpočtu, který nevyžaduje spuštění obvodu. Jakmile jsi spokojený s odhadem času QPU, můžeš Circuit spustit s QESEM.

Při spuštění obvodu provede QESEM protokol charakterizace zařízení přizpůsobený tvému obvodu, čímž vznikne spolehlivý šumový model pro chyby vyskytující se v obvodu. Na základě charakterizace QESEM nejprve implementuje transpilaci s ohledem na šum, která mapuje vstupní Circuit na sadu fyzických Qubitů a Gate, čímž minimalizuje šum ovlivňující cílovou pozorovatelnou veličinu. Patří sem nativně dostupné Gate (CX/CZ na zařízeních IBM&reg;) i další Gate optimalizované QESEM, tvořící rozšířenou sadu Gate systému QESEM. Poté QESEM spustí na QPU sadu ES a EM obvodů založených na charakterizaci a shromáždí výsledky měření. Ty jsou následně klasicky post-procesovány a poskytují nestrannou střední hodnotu a chybové pásmo pro každou pozorovatelnou veličinu odpovídající požadované přesnosti.

![Přehled Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM prokázal schopnost poskytovat vysoce přesné výsledky pro různé kvantové aplikace a pro největší objemy obvodů dosažitelné dnes. QESEM nabízí následující funkce pro uživatele, demonstrované v části s benchmarky níže:
-	**Zaručená přesnost:** QESEM poskytuje nestranné odhady středních hodnot pozorovatelných veličin. Jeho metoda EM je vybavena teoretickými zárukami, které – společně s nejmodernější charakterizací Qedmy – zajišťují, že mitigace konverguje k výstupu bezchybného obvodu s přesností specifikovanou uživatelem. Na rozdíl od mnoha heuristických metod EM, které jsou náchylné k systematickým chybám nebo zkreslení, je zaručená přesnost QESEM nezbytná pro zajištění spolehlivých výsledků u obecných kvantových obvodů a pozorovatelných veličin.
-	**Škálovatelnost na velké QPU:** Čas QPU systému QESEM závisí na objemech obvodů, ale jinak je nezávislý na počtu Qubitů. Qedma demonstrovala QESEM na největších kvantových zařízeních dostupných dnes, včetně IBM Quantum 127-qubitového Eagle a 133-qubitového Heron zařízení.
-	**Nezávislost na aplikaci:** QESEM byl demonstrován na různých aplikacích, včetně Hamiltonovy simulace, VQE, QAOA a amplitudového odhadování. Uživatelé mohou zadat libovolný kvantový Circuit a pozorovatelnou veličinu k měření a získat přesné výsledky bez chyb. Jediná omezení jsou dána hardwarovými specifikacemi a přiděleným časem QPU, které určují přístupné objemy obvodů a přesnosti výstupu. Naproti tomu mnohá řešení pro redukci chyb jsou specifická pro konkrétní aplikaci nebo zahrnují nekontrolované heuristiky, čímž jsou nepoužitelná pro obecné kvantové obvody a aplikace.
-  **Rozšířená sada Gate:** QESEM podporuje Gate s frakčními úhly a poskytuje Qedmou optimalizované frakčně-úhlové $Rzz(\theta)$ Gate na zařízeních IBM Quantum Eagle. Tato rozšířená sada Gate umožňuje efektivnější kompilaci a odemyká objemy obvodů větší až o faktor 2 ve srovnání s výchozí kompilací CX/CZ.
-	**Multibase pozorovatelné veličiny:** QESEM podporuje vstupní pozorovatelné veličiny složené z mnoha nekomutujících Pauliho řetězců, jako jsou obecné Hamiltonovy operátory. Výběr měřicích bází a optimalizace alokace QPU zdrojů (snímky a obvody) je pak prováděna automaticky systémem QESEM tak, aby minimalizovala požadovaný čas QPU pro požadovanou přesnost. Tato optimalizace, která bere v úvahu věrnosti hardwaru a rychlosti spouštění, ti umožňuje spouštět hlubší obvody a dosahovat vyšší přesnosti.
## Benchmarky
QESEM byl testován na široké škále případů použití a aplikací. Následující příklady ti pomohou posoudit, jaké typy pracovních zátěží lze s QESEM spouštět.

Klíčovým ukazatelem pro kvantifikaci náročnosti jak mitigace chyb, tak klasické simulace pro daný Circuit a pozorovatelnou veličinu je **aktivní objem**: počet CNOT Gate ovlivňujících pozorovatelnou veličinu v obvodu. Aktivní objem závisí na hloubce a šířce obvodu, na váze pozorovatelné veličiny a na struktuře obvodu, která určuje světelný kužel pozorovatelné veličiny. Podrobnosti najdeš v přednášce z [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM poskytuje zvláště velkou hodnotu v režimu vysokého objemu, kde dává spolehlivé výsledky pro obecné obvody a pozorovatelné veličiny.

![Aktivní objem](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplikace    | Počet Qubitů | Zařízení | Popis obvodu | Přesnost | Celkový čas | Využití Runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE Circuit  | 8              | Eagle (r3) | 21 celkových vrstev, 9 měřicích bází, 1D řetězec                    | 98 %      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 jedinečné vrstvy × 3 kroky, 2D topologie heavy-hex                      | 97 %     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 jedinečné vrstvy × 8 kroků, 2D topologie heavy-hex                     | 97 %      | 116 min      | 23 min          |
| Trotterizovaná Hamiltonova simulace   | 40  | Eagle (r3)            | 2 jedinečné vrstvy × 10 Trotterových kroků, 1D řetězec                    | 97 %      | 3 hodiny     | 25 min         |
| Trotterizovaná Hamiltonova simulace   | 119  | Eagle (r3)           | 3 jedinečné vrstvy × 9 Trotterových kroků, 2D topologie heavy-hex                    | 95 %      | 6,5 hodiny     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 jedinečné vrstvy × 15 kroků, 2D topologie heavy-hex                    | 99 %      | 52 min             | 9 min           |

Přesnost je zde měřena relativně k ideální hodnotě pozorovatelné veličiny: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, kde '$\epsilon$' je absolutní přesnost mitigace (nastavená vstupem uživatele) a $\langle O \rangle_{ideal}$ je pozorovatelná veličina bezchybného obvodu.
„Využití Runtime" měří využití benchmarku v dávkovém režimu (součet využití jednotlivých úloh), zatímco „celkový čas" měří využití v session režimu (celková doba experimentu), která zahrnuje dodatečné klasické a komunikační časy. QESEM je dostupný pro spouštění v obou režimech, takže uživatelé mohou co nejlépe využít své dostupné zdroje.

28-qubitové obvody Kicked Ising simulují Diskrétní časový kvazikrystal studovaný Shinjem et al. (viz [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) a [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) na třech propojených smyčkách ibm_kawasaki. Parametry obvodu použité zde jsou $(\theta_x, \theta_z) = (0.9 \pi, 0)$ s feromagnetickým počátečním stavem $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Měřená pozorovatelná veličina je absolutní hodnota magnetizace $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Experiment Kicked Ising v měřítku užitečnosti byl spuštěn na 136 nejlepších Qubitech ibm_fez; tento konkrétní benchmark byl spuštěn na Cliffordově úhlu $(\theta_x, \theta_z) = (\pi, 0)$, při němž aktivní objem roste s hloubkou obvodu pomalu, což – společně s vysokou věrností zařízení – umožňuje vysokou přesnost při krátkém čase běhu.

Trotterizované obvody Hamiltonovy simulace jsou pro model Ising s příčným polem při frakčních úhlech: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ a $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ (viz [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Obvod v měřítku užitečnosti byl spuštěn na 119 nejlepších Qubitech ibm_brisbane, zatímco experiment se 40 Qubity byl spuštěn na nejlepším dostupném řetězci. Přesnost je uváděna pro magnetizaci; vysoce přesné výsledky byly získány i pro pozorovatelné veličiny s vyšší vahou.

Obvod VQE byl vyvinut společně s výzkumníky z Centra pro kvantové technologie a aplikace Německého elektronu-synchrotronu (DESY). Cílová pozorovatelná veličina zde byl Hamiltonův operátor složený z velkého počtu nekomutujících Pauliho řetězců, což zdůrazňuje optimalizovaný výkon QESEM pro pozorovatelné veličiny ve více bázích. Mitigace byla aplikována na klasicky optimalizovaný ansatz; ačkoli tyto výsledky jsou zatím nepublikované, výsledky stejné kvality budou získány pro různé obvody s podobnými strukturálními vlastnostmi.
## Začínáme
Autentizuj se pomocí svého [IBM Quantum Platform API klíče](http://quantum.cloud.ibm.com/) a vyber Qiskit Function QESEM takto. (Tento úryvek předpokládá, že jsi již [uložil svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do svého lokálního prostředí.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Příklad
Pro začátek vyzkoušej tento základní příklad odhadování požadovaného času QPU pro spuštění QESEM pro daný `pub`:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Následující příklad spustí úlohu QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Pro kontrolu stavu pracovní zátěže Qiskit Function nebo získání výsledků můžeš použít známá API Qiskit Serverless:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Parametry funkce
| Název |  Typ | Popis | Povinný | Výchozí |  Příklad |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) | Toto je hlavní vstup. `Pub` obsahuje 2–4 prvky: Circuit, jeden nebo více pozorovatelných veličin, 0 nebo jednu sadu hodnot parametrů a volitelnou přesnost. Pokud přesnost nebyla zadána, použije se `default_precision` z `options`. |  Ano |  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string | Název Backend, který se má použít | Ne | QESEM automaticky vybere nejméně vytížené zařízení hlášené IBM | `"ibm_fez"` |
| `instance` | string | Název cloudového zdroje (CRN) instance, která se má použít, v daném formátu | Ne | N/A | `"CRN"` |
| `options` | dictionary | Vstupní možnosti. Více podrobností najdeš v části **Možnosti**. | Ne | Viz část **Možnosti**. | `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}` |

### Možnosti
| Možnost |  Volby | Popis | Výchozí |
| -----| -----------| -------- | ------- |
| `estimate_time_only` | `"analytical"` / `"empirical"` / None | Pokud je nastaveno, úloha QESEM pouze vypočítá odhad času QPU. Více podrobností viz následující popis. Pokud je nastaveno na None, Circuit se spustí pomocí QESEM. | None |
| `default_precision` | 0 < float | Platí pro `pubs`, které nemají zadanou přesnost. Přesnost označuje přijatelnou chybu očekávaných hodnot pozorovatelných veličin v absolutní hodnotě. Konkrétně: doba běhu QPU pro mitigaci bude určena tak, aby výstupní hodnoty všech sledovaných pozorovatelných veličin spadaly do `1σ` intervalu spolehlivosti cílové přesnosti. Pokud je zadáno více pozorovatelných veličin, mitigace poběží, dokud nebude dosažena cílová přesnost pro každou z nich. | 0.02 |
| `max_execution_time` | 0 < integer < 28 800 (8 hodin) | Umožňuje ti omezit dobu QPU v sekundách, která se použije pro celý proces QESEM. Více podrobností viz níže. | 3 600 (jedna hodina) |
| `transpilation_level` | 0 / 1 | Viz popis níže. | 1 |
| `execution_mode` | `"session"` / `"batch"` | Viz následující popis. | "batch" |

> **Caution:** Odhad doby QPU se liší podle Backend. Proto při spouštění funkce QESEM dbej na to, aby byl spuštěn na stejném Backend, který byl vybrán při získávání odhadu doby QPU.

> **Note:** QESEM ukončí svůj běh, jakmile dosáhne cílové přesnosti nebo jakmile dosáhne `max_execution_time`, podle toho, co nastane dříve.

- `estimate_time_only` – Tento příznak umožňuje uživatelům získat odhad doby QPU potřebné k provedení Circuit pomocí QESEM.
    - Pokud je nastaveno na `"analytical"`, vypočítá se horní mez doby QPU bez spotřeby jakéhokoli QPU. Tento odhad má rozlišení 30 minut (například 30 minut, 60 minut, 90 minut atd.). Bývá zpravidla pesimistický a lze ho získat pouze pro jednotlivé Pauliho pozorovatelné veličiny nebo součty Pauliho operátorů bez překrývajících se nosičů (například Z0+Z1). Hodí se především pro porovnání složitosti různých parametrů zadaných uživatelem (Circuit, přesnost atd.).
    - Pro přesnější odhad doby QPU nastav tento příznak na `"empirical"`. Ačkoli tato možnost vyžaduje spuštění malého počtu Circuit, poskytuje výrazně přesnější odhad doby QPU. Tento odhad má rozlišení 5 minut (například 20 minut, 25 minut, 30 minut atd.). Uživatel může zvolit spuštění empirického odhadu doby v batch nebo session režimu. Více podrobností viz popis `execution_mode`. Například v batch režimu empirický odhad doby spotřebuje méně než 10 minut QPU.

- `max_execution_time`: Umožňuje ti omezit dobu QPU v sekundách, která se použije pro celý proces QESEM. Protože konečná doba QPU potřebná k dosažení cílové přesnosti je určována dynamicky v průběhu úlohy QESEM, tento parametr ti umožňuje omezit náklady experimentu. Pokud je dynamicky určená doba QPU kratší než čas přidělený uživatelem, tento parametr nebude mít na experiment vliv. Parametr `max_execution_time` je zvláště užitečný v případech, kdy je analytický odhad doby poskytnutý QESEM před spuštěním úlohy příliš pesimistický a uživatel chce přesto zahájit mitigační úlohu. Po dosažení časového limitu přestane QESEM odesílat nové Circuit. Circuit, které již byly odeslány, dál poběží (takže celkový čas může překročit limit až o 30 minut), a uživatel obdrží zpracované výsledky z Circuit spuštěných do té doby. Pokud chceš uplatnit limit doby QPU kratší, než je analytický odhad doby, poraď se s Qedmou a získej odhad přesnosti dosažitelné v rámci tohoto časového limitu.

- `transpilation_level`: Po odeslání Circuit do QESEM automaticky připraví několik alternativních transpilací Circuit a vybere tu, která minimalizuje dobu QPU. Například alternativní transpilace mohou využívat Qedmou optimalizované frakční RZZ Gate ke snížení hloubky Circuit. Samozřejmě všechny transpilace jsou ekvivalentní vstupnímu Circuit z hlediska jejich ideálního výstupu. Pro větší kontrolu nad transpilací Circuit nastav úroveň transpilace v `options`. Zatímco `"transpilation_level": 1` odpovídá výchozímu chování popsanému výše, `"transpilation_level": 0` zahrnuje pouze minimální úpravy původního Circuit; například „layerifikaci" – organizaci operací Circuit do „vrstev" simultánních dvou-Qubitových Gate. Poznamenejme, že automatické mapování na hardware s vysokou věrností Qubitů se provádí v každém případě.

| transpilation_level | Popis |
|:-:|:--|
| `1` | Výchozí transpilace QESEM. Připraví několik alternativních transpilací a vybere tu, která minimalizuje dobu QPU. Bariéry mohou být upraveny v kroku layerifikace. |
| `0` | Minimální transpilace: mitigovaný Circuit bude strukturálně blízký vstupnímu Circuit. Circuit poskytnuté na úrovni 0 by měly odpovídat konektivitě zařízení a měly by být zadány pomocí následujících Gate: CX, Rzz(α) a standardních jednoqubitových Gate (U, x, sx, rz atd.). Bariéry budou respektovány v kroku layerifikace. |

- `execution_mode` – Uživatel si může vybrat, zda spustí úlohu QESEM v dedikované [IBM session](/guides/execution-modes#session-mode) nebo napříč více [IBM batch](/guides/execution-modes#batch-mode):
    - **Session Mode**: Session jsou dražší, ale vedou ke kratší době do výsledku. Jakmile session začne, QPU je rezervováno výhradně pro úlohu QESEM. Výpočet využití zahrnuje jak čas strávený prováděním na QPU, tak související klasické výpočty (prováděné QESEM a IBM). Qiskit Function QESEM se stará o vytvoření a ukončení session automaticky. Uživatelům s neomezeným přístupem k QPU (například při on-premises nasazení) se doporučuje používat session mode pro rychlejší provádění QESEM.
    - **Batch Mode**: V batch mode je QPU uvolněno během klasických výpočtů, což vede k nižšímu využití QPU. Protože batch úlohy typicky trvají delší dobu, existuje větší riziko hardwarových driftů; QESEM zahrnuje opatření pro detekci a kompenzaci driftů, čímž zachovává spolehlivost během delších běhů.

> **Note:** Operace bariéry se typicky používají k určení vrstev dvou-Qubitových Gate v kvantových Circuit. Na úrovni 0 QESEM zachovává vrstvy určené bariérami. Na úrovni 1 jsou vrstvy určené bariérami považovány za jednu alternativu transpilace při minimalizaci doby QPU.

### Výstupy
Výstupem Circuit funkce je [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), který obsahuje dvě pole:

- Jeden objekt [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Lze ho indexovat přímo z `PrimitiveResult`.

- Metadata na úrovni úlohy.

Každý `PubResult` obsahuje pole `data` a `metadata`.

- Pole `data` obsahuje alespoň pole očekávaných hodnot (`PubResult.data.evs`) a pole standardních chyb (`PubResult.data.stds`). Může také obsahovat další data v závislosti na použitých možnostech.

- Pole `metadata` obsahuje metadata na úrovni PUB (`PubResult.metadata`).

Následující úryvek kódu popisuje, jak získat odhad doby QPU (když je nastaveno `estimate_time_only`):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

Následující úryvek kódu ukazuje, jak získat výsledky mitigace (pokud není nastaveno `estimate_time_only`) a metriky provádění. Tyto obsahují základní data, která umožňují hlubší pochopení toho, jak různé parametry ovlivňují provádění QESEM. Mohou být také relevantní při psaní článku na základě tvého výzkumu.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Načtení chybových zpráv
Pokud má tvoje úloha stav ERROR, použij `job.result()` k načtení chybové zprávy takto:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>